# import packages

In [1]:
import pandas as pd
import xarray as xr
import geopandas as gpd
import glob
import os
import numpy as np
from datetime import datetime
import re

# process hourly AORC forcings into ngen compliant format

In [4]:
for file in np.sort(glob.glob('/beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_term/raw/aorc.*.camels_ngen_hf.v2_2.nc')):
    ds = xr.open_dataset(file)
    file_name = os.path.basename(file)
    print(f'processing {file_name}')

    # Add a new variable 'ids' of type string, using the values of the 'catchment-id' coordinate
    ds = ds.rename({'hru': 'catchment-id'})
    ds = ds.assign({'ids': ('catchment-id', ds['catchment-id'].values)})
    ds = ds.transpose('catchment-id', 'time')
    
    # Update time dimensions to Unix epoch
    reference_time = np.datetime64('1970-01-01 00:00:00')
    time_since = ((ds['time'] - reference_time) / np.timedelta64(1, "h")).astype('int')
    
    # Create 2D Time variable
    time_2d = time_since.expand_dims({'catchment-id': ds.dims['catchment-id']}, axis=0)
    ds['Time'] = time_2d
    ds['Time'].attrs['units'] = 'h'
    #ds['Time'].attrs['epoch_start'] = '1970-01-01 00:00:00'
    ds['Time'].attrs['epoch_start'] = '01/01/70 00:00:00'
    
    # Set global epoch_start
    #ds.attrs['epoch_start'] = '1970-01-01 00:00:00'
    ds.attrs['epoch_start'] = '01/01/70 00:00:00'
    
    # Change double to float32
    for var in ds.variables:
        if ds[var].dtype == np.float64:
            ds[var] = ds[var].astype(np.float32)

    # Save the updated dataset to a new NetCDF file
    output_path = '/beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_term/ngen_fmt/' + file_name
    if os.path.exists(output_path):
        os.remove(output_path)
    #ds_slice = ds.sel(time=slice('2000-01-01','2000-01-31'))
    ds.to_netcdf(output_path)

    print(f"  Variables renamed and saved to {output_path}")
    # break


processing aorc.1979.camels_ngen_hf.v2_2.nc
  Variables renamed and saved to /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_term/ngen_fmt/aorc.1979.camels_ngen_hf.v2_2.nc
processing aorc.1980.camels_ngen_hf.v2_2.nc
  Variables renamed and saved to /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_term/ngen_fmt/aorc.1980.camels_ngen_hf.v2_2.nc
processing aorc.1981.camels_ngen_hf.v2_2.nc
  Variables renamed and saved to /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_term/ngen_fmt/aorc.1981.camels_ngen_hf.v2_2.nc
processing aorc.1982.camels_ngen_hf.v2_2.nc
  Variables renamed and saved to /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_term/ngen_fmt/aorc.1982.camels_ngen_hf.v2_2.nc
processing aorc.1983.camels_ngen_hf.v2_2.nc
  Variables renamed and saved to /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_term/ngen_fmt/aorc.1983.camels_ngen_h

# resample to daily

In [2]:
for file in np.sort(glob.glob('/beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/hourly/incl_terminal/ngen_fmt/aorc.*.camels_ngen_hf.v2_2.nc')):
    ds = xr.open_dataset(file)
    
    # Extract the year using regex
    in_file_name = os.path.basename(file)
    match = re.search(r'aorc\.(\d{4})\.', in_file_name)
    if match:
        year = match.group(1)
        #print(f"File: {in_file_name}, Year: {year}")
        
    out_file_name = f'aorc_daily.{year}.camels_ngen_hf.v2_2.nc'
    print(f'processing {out_file_name}')

    # get daily mean - and temp min/max, APCP_surface sum
    ds_daily = ds.resample(time='D').mean()
    ds_daily['TMPmin_2maboveground'] = ds.TMP_2maboveground.resample(time='D').min()
    ds_daily['TMPmax_2maboveground'] = ds.TMP_2maboveground.resample(time='D').max()
    ds_daily['APCP_surface'] = ds.APCP_surface.resample(time='D').sum()
    
    # Update time dimensions to Unix epoch
    reference_time = np.datetime64('1970-01-01 00:00:00')
    time_since = ((ds_daily['time'] - reference_time) / np.timedelta64(1, "h")).astype('int')
    
    # Create 2D Time variable
    time_2d = time_since.expand_dims({'catchment-id': ds_daily.dims['catchment-id']}, axis=0)
    ds_daily['Time'] = time_2d
    ds_daily['Time'].attrs['units'] = 'h'
    #ds_daily['Time'].attrs['epoch_start'] = '1970-01-01 00:00:00'
    ds_daily['Time'].attrs['epoch_start'] = '01/01/70 00:00:00'
    
    # Set global epoch_start
    #ds_daily.attrs['epoch_start'] = '1970-01-01 00:00:00'
    ds_daily.attrs['epoch_start'] = '01/01/70 00:00:00'
    
    # Extract unique values from the 'ids' variable along the 'catchment-id' dimension
    ids_catchment = ds_daily['ids'].isel(time=0)  # Select one time slice (since it duplicates across time)

    # Add the updated variable back to the dataset, or create a new dataset
    ds_daily = ds_daily.drop_vars('ids').assign(ids=ids_catchment)
    
    # save out
    out_file = '/beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/daily/incl_terminal/' + out_file_name
    print(f'  saving out {out_file}')
    if os.path.exists(out_file):
        os.remove(out_file)
    ds_daily.to_netcdf(out_file)
    
    # for debugging
    # break


processing aorc_daily.1979.camels_ngen_hf.v2_2.nc
  saving out /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/daily/incl_terminal/aorc_daily.1979.camels_ngen_hf.v2_2.nc
processing aorc_daily.1980.camels_ngen_hf.v2_2.nc
  saving out /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/daily/incl_terminal/aorc_daily.1980.camels_ngen_hf.v2_2.nc
processing aorc_daily.1981.camels_ngen_hf.v2_2.nc
  saving out /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/daily/incl_terminal/aorc_daily.1981.camels_ngen_hf.v2_2.nc
processing aorc_daily.1982.camels_ngen_hf.v2_2.nc
  saving out /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/daily/incl_terminal/aorc_daily.1982.camels_ngen_hf.v2_2.nc
processing aorc_daily.1983.camels_ngen_hf.v2_2.nc
  saving out /beegfs/projects/aw-ciroh/common/lm_forcing/aorc/camels_ngen_v2_2/daily/incl_terminal/aorc_daily.1983.camels_ngen_hf.v2_2.nc
processing aorc_daily.1984.camels_ngen_hf.v2_2.nc
  saving o